# 01 — Exploratory Data Analysis (EDA)

In this notebook we're going to explore the **COCO val2017** dataset and get
a feel for what we're working with — especially the object density per image,
which is the main thing we care about for high-density segmentation.

Here's what we'll look at:
1. Basic dataset stats
2. How objects are distributed across images
3. Which categories show up the most
4. Some actual sample images at different density levels
5. Challenges we noticed that'll probably hurt our baseline

---
## Section 1 — Dataset Overview

Let's start by loading up the COCO annotations and seeing what we've got.
COCO val2017 has 5,000 images with 80 object categories — it's a pretty
standard benchmark so this should give us a solid starting point.

In [1]:
import sys, os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from PIL import Image
from collections import Counter
from tqdm import tqdm

# Add project root to path so we can import our modules
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.data_loader import get_dense_images, load_image_and_masks, get_dataset_stats

# Make sure output dirs exist
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load COCO annotations
ann_file = '../data/annotations/annotations/instances_val2017.json'
print(f'Loading annotations from {ann_file} ...')
coco = COCO(ann_file)
print()

# Get dataset stats
stats = get_dataset_stats(coco)
print()
print(f'Number of unique categories: {stats["num_categories"]}')

Loading annotations from ../data/annotations/annotations/instances_val2017.json ...
loading annotations into memory...


Done (t=0.37s)


creating index...
index created!

Total images       : 5000
Total annotations  : 36335
Unique categories  : 80
Avg objects/image  : 7.27
Min objects/image  : 0
Max objects/image  : 62

Number of unique categories: 80


---
## Section 2 — Object Density Distribution

This is the key part — we need to understand how many objects each image has.
Images with lots of objects are way harder to segment because things overlap
and get crowded together.

We'll make two plots:
- A histogram of object counts per image
- A bar chart splitting images into 4 density groups: 1-5, 5-15, 15-30, 30-50 objects

In [3]:
# Count objects per image
img_ids = coco.getImgIds()
obj_counts = []
for img_id in tqdm(img_ids, desc='Counting objects per image'):
    ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=False)
    obj_counts.append(len(ann_ids))

obj_counts = np.array(obj_counts)
print(f'Computed counts for {len(obj_counts)} images.')
print(f'Mean: {obj_counts.mean():.2f}, Median: {np.median(obj_counts):.1f}, '
      f'Std: {obj_counts.std():.2f}')

Counting objects per ima

Counting objects per ima

Computed counts for 5000 images.
Mean: 7.27, Median: 4.0, Std: 7.25


In [4]:
# Plot histogram + density group bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(obj_counts, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Number of Objects per Image', fontsize=12)
axes[0].set_ylabel('Number of Images', fontsize=12)
axes[0].set_title('Distribution of Object Counts per Image', fontsize=14, fontweight='bold')
axes[0].axvline(obj_counts.mean(), color='red', linestyle='--', label=f'Mean = {obj_counts.mean():.1f}')
axes[0].legend(fontsize=11)

# Density groups
groups = {
    '1–5 (Low)': np.sum((obj_counts >= 1) & (obj_counts <= 5)),
    '5–15 (Medium)': np.sum((obj_counts > 5) & (obj_counts <= 15)),
    '15–30 (High)': np.sum((obj_counts > 15) & (obj_counts <= 30)),
    '30–50 (Very High)': np.sum((obj_counts > 30) & (obj_counts <= 50)),
}
colors = ['#4CAF50', '#FF9800', '#F44336', '#9C27B0']
bars = axes[1].bar(groups.keys(), groups.values(), color=colors, edgecolor='white')
axes[1].set_xlabel('Density Group', fontsize=12)
axes[1].set_ylabel('Number of Images', fontsize=12)
axes[1].set_title('Images by Object Density Group', fontsize=14, fontweight='bold')
for bar, val in zip(bars, groups.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                str(val), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/density_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/density_distribution.png')

for name, count in groups.items():
    print(f'  {name}: {count} images')

Saved: results/figures/density_distribution.png
  1–5 (Low): 2769 images
  5–15 (Medium): 1541 images
  15–30 (High): 577 images
  30–50 (Very High): 60 images


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_67743/2613363120.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 3 — Top Categories

COCO has 80 categories but they're not evenly spread out at all.
Some categories like **person** have thousands of annotations while others barely show up.
This kind of class imbalance is something we'll need to keep in mind —
it could mess with any learning-based approach we try later.

In [5]:
# Count annotations per category
cat_ids = coco.getCatIds()
cats = coco.loadCats(cat_ids)
cat_names = {cat['id']: cat['name'] for cat in cats}

ann_ids_all = coco.getAnnIds(iscrowd=False)
anns_all = coco.loadAnns(ann_ids_all)

cat_counter = Counter()
for ann in tqdm(anns_all, desc='Counting categories'):
    cat_counter[cat_names[ann['category_id']]] += 1

top20 = cat_counter.most_common(20)
names = [item[0] for item in top20]
counts = [item[1] for item in top20]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(names[::-1], counts[::-1], color='#2196F3', edgecolor='white')
ax.set_xlabel('Number of Annotations', fontsize=12)
ax.set_title('Top 20 Most Frequent Object Categories (COCO val2017)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/category_distribution.png')

Counting categories:   0

Counting categories: 100

Saved: results/figures/category_distribution.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_67743/2923516410.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 4 — Visual Samples

Looking at pictures is honestly way more helpful than just staring at numbers.
We'll pick 6 images — 2 low density, 2 medium, 2 high — and draw the COCO
masks on top so we can actually see what we're dealing with.

In [6]:
import random
random.seed(42)

img_dir = '../data/images/val2017'

# Split images into density groups
low_ids = [img_id for img_id, c in zip(img_ids, obj_counts) if 1 <= c <= 5]
med_ids = [img_id for img_id, c in zip(img_ids, obj_counts) if 10 <= c <= 20]
hi_ids  = [img_id for img_id, c in zip(img_ids, obj_counts) if c >= 30]

print(f'Low density (1-5):    {len(low_ids)} images')
print(f'Medium density (10-20): {len(med_ids)} images')
print(f'High density (30+):  {len(hi_ids)} images')

sample_low = random.sample(low_ids, 2)
sample_med = random.sample(med_ids, 2)
sample_hi  = random.sample(hi_ids, 2)
sample_ids = sample_low + sample_med + sample_hi
labels = ['Low', 'Low', 'Medium', 'Medium', 'High', 'High']

print(f'\nSelected image IDs: {sample_ids}')

Low density (1-5):    2769 images
Medium density (10-20): 1014 images
High density (30+):  71 images

Selected image IDs: [299609, 476215, 559842, 474293, 316666, 546219]


In [7]:
# Show samples with overlaid masks
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (img_id, label) in enumerate(zip(sample_ids, labels)):
    ax = axes[idx // 3][idx % 3]
    image, anns = load_image_and_masks(coco, img_id, img_dir)
    ax.imshow(image)
    for ann in anns:
        if 'segmentation' in ann and not ann.get('iscrowd', False):
            color = np.random.random(3)
            if isinstance(ann['segmentation'], list):
                for seg in ann['segmentation']:
                    poly = np.array(seg).reshape(-1, 2)
                    ax.fill(poly[:, 0], poly[:, 1], alpha=0.3, color=color)
                    ax.plot(poly[:, 0], poly[:, 1], linewidth=1, color=color)
    ax.set_title(f'{label} density — {len(anns)} objects (ID: {img_id})',
                 fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Images at Different Density Levels (with COCO masks)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/sample_images.png')

Saved: results/figures/sample_images.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_67743/1115183317.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 5 — Key Observations and Challenges

After going through the data, here are the main challenges we found that
are probably going to give our baseline models a hard time:

### 1. Heavy Occlusion Between Objects
In the high-density images (30+ objects), stuff overlaps all the time.
You can't really tell where one object ends and another starts. Visualizing
the heavy occlusion here was honestly pretty tricky. Classical methods like
Watershed need clear brightness gradients between objects, and those just
disappear when things overlap — so it'll probably merge a bunch of objects together.

### 2. Scale Variation (Tiny vs Large Objects in the Same Image)
Some images have both really tiny objects and huge ones. Like a tiny fork
sitting on a large table. If we use a fixed blur kernel (which we do in Watershed),
it'll smooth out the small objects completely while barely affecting the big ones.
There's no good way around this without some kind of multi-scale approach.

### 3. Class Imbalance Across Categories
The category distribution is super skewed — **person** has way more annotations
than most other categories combined. Our baselines don't really care about categories
since they're just looking at pixel values, but if we move to learning-based methods
later this imbalance will definitely be an issue.

### 4. Objects Cut Off at Image Boundaries
A lot of objects are only partially visible because they extend past the edge
of the image. These truncated objects have weird incomplete masks which will
confuse region-growing methods. Connected components will just treat them as
complete objects and give us wrong counts.

---
**Bottom line:** These four things — occlusion, scale variation, class imbalance,
and boundary truncation — basically explain why we expect classical methods to
do poorly on dense images. We'll see exactly how bad it gets in Notebook 02.

---
## Save Statistics

Saving everything to a JSON file so we can reference it in the report.

In [8]:
# Save all EDA stats to JSON
eda_stats = {
    'total_images': stats['total_images'],
    'total_annotations': stats['total_annotations'],
    'num_categories': stats['num_categories'],
    'avg_objects_per_image': round(stats['avg_objects'], 2),
    'min_objects_per_image': stats['min_objects'],
    'max_objects_per_image': stats['max_objects'],
    'density_groups': {
        '1-5_low': int(np.sum((obj_counts >= 1) & (obj_counts <= 5))),
        '5-15_medium': int(np.sum((obj_counts > 5) & (obj_counts <= 15))),
        '15-30_high': int(np.sum((obj_counts > 15) & (obj_counts <= 30))),
        '30-50_very_high': int(np.sum((obj_counts > 30) & (obj_counts <= 50))),
    },
    'top_10_categories': dict(Counter({cat_names[ann['category_id']]: 0 for ann in anns_all})),
}

# Fix top 10 — take actual top 10
eda_stats['top_10_categories'] = dict(cat_counter.most_common(10))

with open('../results/metrics/eda_stats.json', 'w') as f:
    json.dump(eda_stats, f, indent=2)

print('Saved: results/metrics/eda_stats.json')
print(json.dumps(eda_stats, indent=2))

Saved: results/metrics/eda_stats.json
{
  "total_images": 5000,
  "total_annotations": 36335,
  "num_categories": 80,
  "avg_objects_per_image": 7.27,
  "min_objects_per_image": 0,
  "max_objects_per_image": 62,
  "density_groups": {
    "1-5_low": 2769,
    "5-15_medium": 1541,
    "15-30_high": 577,
    "30-50_very_high": 60
  },
  "top_10_categories": {
    "person": 10777,
    "car": 1918,
    "chair": 1771,
    "book": 1129,
    "bottle": 1013,
    "cup": 895,
    "dining table": 695,
    "traffic light": 634,
    "bowl": 623,
    "handbag": 540
  }
}
